In [ ]:
import polars as pl
import json

In [39]:
YOUR_ID = 'user' + '586545531'
YOUR_USERNAME = 'Maximusin'

In [64]:
with open('./data/result.json', 'r', encoding='utf-8') as f:
    tg_json = json.load(f)

data = []

for message in tg_json['messages']:
    msg_type = message.get('type')
    date = message.get('date')
    sender = message.get('from') or message.get('actor') or message.get('from_id', 'Unknown')
    
    raw_text = message.get('text', '')
    if isinstance(raw_text, str):
        text = raw_text
    elif isinstance(raw_text, list):
        parts = []
        for part in raw_text:
            if isinstance(part, dict):
                parts.append(part.get('text', ''))
            else:
                parts.append(str(part))
        text = ''.join(parts)
    else:
        text = ''
    
    data.append({
        "msg_type": msg_type,
        "date": date,
        "sender": sender,
        "text": text
    })
    
df = pl.DataFrame(data)
df = df.with_columns(pl.col('date').str.to_datetime())
df = df.filter(pl.col('text') != '', pl.col('msg_type') == 'message')

df = df.with_columns(
    pl.when(
    (pl.col('sender') == YOUR_ID) | (pl.col('sender') == YOUR_USERNAME)
    ).then(pl.lit('gpt')).otherwise(pl.lit('human')).alias('role'))

df = df.with_columns(pl.col('date').diff().alias('time_delta'))

df = df.with_columns(
    pl.when(
        (pl.col('role') != pl.col('role').shift()) | (pl.col('time_delta').dt.total_seconds() > 120)
    )
    .then(True)
    .otherwise(False)
    .alias('break')
)

df = df.with_columns(pl.col('break').cast(pl.Int64).cum_sum().alias('group_id'))